# CodeTune v2 — HumanEval Evaluation

**Environment**: Google Colab (L4 / A100 80GB)  
**Models**: base → sft_A → sft_C  

只需上传这一个文件，所有依赖自动安装，结果保存到 Drive。

In [ ]:
import subprocess, torch
result = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True)
print("GPU:", result.stdout.strip())
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB, BF16: {torch.cuda.is_bf16_supported()}")

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes datasets huggingface-hub click tqdm -q
print("Done.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = "/content/drive/MyDrive/codetune"
RESULTS_DIR = f"{DRIVE_ROOT}/eval_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Drive mounted. Results →", RESULTS_DIR)

In [ ]:
from huggingface_hub import login
HF_TOKEN = ""   # ← 填入 HF token
login(token=HF_TOKEN)
print("HF login OK")

In [ ]:
# ── 评估核心代码（内联，无需上传任何脚本）────────────────────────────
from __future__ import annotations
import json, re, shutil, subprocess, sys, tempfile
from pathlib import Path
from tqdm import tqdm
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are an expert Python programmer. Complete the given function. "
    "Write only the function body — no extra explanation, no test code."
)
BASE_MODEL = "unsloth/Qwen3.5-9B"


def _clean_completion(text: str, prompt: str) -> str:
    text = text.strip()
    text = re.sub(r'^```(?:python)?\s*\n', '', text)
    text = re.sub(r'\n?```\s*$', '', text)
    text = text.strip('\n')

    # Strip <think>...</think> blocks (Qwen3 reasoning)
    text = re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL)
    text = text.strip('\n')

    # If model repeated the full function definition, extract only the body
    def_match = re.search(r'^def\s+\w+\(', prompt, re.MULTILINE)
    if def_match:
        func_name = re.search(r'def\s+(\w+)', def_match.group()).group(1)
        pattern = rf'def\s+{re.escape(func_name)}\s*\(.*?\n'
        def_in_completion = re.search(pattern, text, re.DOTALL)
        if def_in_completion:
            after_def = text[def_in_completion.end():]
            doc_match = re.match(r'\s*""".*?"""\s*\n', after_def, re.DOTALL)
            text = after_def[doc_match.end():] if doc_match else after_def

    text = text.strip('\n')

    lines = text.splitlines()
    first_nonempty = next((l for l in lines if l.strip()), None)
    if first_nonempty and not first_nonempty.startswith("    "):
        text = "\n".join("    " + l if l.strip() else l for l in lines)

    return text


def _is_adapter_only(model_path: str) -> bool:
    p = Path(model_path)
    if p.exists():
        return not (p / "config.json").exists()
    try:
        from huggingface_hub import list_repo_files
        files = list(list_repo_files(model_path))
        return "adapter_config.json" in files and "config.json" not in files
    except Exception:
        return False


def _fix_adapter_keys(adapter_dir: str) -> str:
    from safetensors.torch import load_file, save_file
    src = Path(adapter_dir)
    weights = load_file(str(src / "adapter_model.safetensors"))
    new_weights = {}
    for k, v in weights.items():
        nk = k.replace(".language_model.", ".")
        nk = nk.replace(".lora_A.weight", ".lora_A.default.weight")
        nk = nk.replace(".lora_B.weight", ".lora_B.default.weight")
        new_weights[nk] = v
    tmp = tempfile.mkdtemp()
    shutil.copy(str(src / "adapter_config.json"), f"{tmp}/adapter_config.json")
    save_file(new_weights, f"{tmp}/adapter_model.safetensors")
    print(f"Adapter keys remapped → {tmp}")
    return tmp


def _resolve_adapter(model_path: str) -> str:
    p = Path(model_path)
    if p.exists():
        return _fix_adapter_keys(model_path)
    else:
        from huggingface_hub import snapshot_download
        print(f"Downloading adapter from HF: {model_path}")
        local = snapshot_download(model_path)
        return _fix_adapter_keys(local)


def _get_text_tokenizer(tokenizer):
    """Extract pure text tokenizer from a multimodal Processor if needed."""
    if hasattr(tokenizer, 'tokenizer'):
        # It's a multimodal Processor (e.g. Qwen-VL) — extract text tokenizer
        print("Multimodal Processor detected — extracting text tokenizer")
        return tokenizer.tokenizer
    return tokenizer


def load_model(model_path: str):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

    if _is_adapter_only(model_path):
        print(f"LoRA adapter detected — loading {BASE_MODEL} + adapter")
        tokenizer = _get_text_tokenizer(AutoTokenizer.from_pretrained(BASE_MODEL))
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, quantization_config=bnb_config, device_map="auto")
        fixed = _resolve_adapter(model_path)
        model = PeftModel.from_pretrained(model, fixed)
    else:
        tokenizer = _get_text_tokenizer(AutoTokenizer.from_pretrained(model_path))
        model = AutoModelForCausalLM.from_pretrained(
            model_path, quantization_config=bnb_config, device_map="auto")

    model.eval()
    return model, tokenizer


def generate_completion(model, tokenizer, prompt: str, max_new_tokens: int = 512) -> str:
    import torch
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Complete this Python function:\n\n{prompt}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Use encode() instead of __call__() to bypass multimodal image-processing path
    input_ids = tokenizer.encode(text, return_tensors="pt",
                                 truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    return _clean_completion(raw, prompt)


def run_tests(prompt: str, completion: str, test_code: str, entry_point: str, timeout: int = 10) -> dict:
    full_code = prompt + completion + "\n\n" + test_code + f"\n\ncheck({entry_point})\n"
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, encoding="utf-8") as f:
        f.write(full_code)
        tmp = f.name
    try:
        r = subprocess.run([sys.executable, tmp], capture_output=True, text=True, timeout=timeout)
        return {"passed": r.returncode == 0, "stderr": r.stderr[:400]}
    except subprocess.TimeoutExpired:
        return {"passed": False, "stderr": "TIMEOUT"}
    except Exception as e:
        return {"passed": False, "stderr": str(e)}
    finally:
        Path(tmp).unlink(missing_ok=True)


def evaluate(model_path: str, label: str, subset=None):
    print("Loading HumanEval …")
    he_ds = load_dataset("openai/openai_humaneval", split="test")
    problems = list(he_ds)
    if subset:
        ids = {int(x) for x in str(subset).split(",")}
        problems = [p for p in problems if int(p["task_id"].split("/")[-1]) in ids]
        print(f"Subset: {[p['task_id'] for p in problems]}")
    print(f"Problems: {len(problems)}")

    print(f"Loading model: {model_path}")
    model, tokenizer = load_model(model_path)

    results, passed_count = [], 0
    for prob in tqdm(problems, desc=f"[{label}]", unit="prob"):
        completion  = generate_completion(model, tokenizer, prob["prompt"])
        test_result = run_tests(prob["prompt"], completion, prob["test"], prob["entry_point"])
        if test_result["passed"]:
            passed_count += 1
        results.append({
            "task_id":    prob["task_id"],
            "passed":     test_result["passed"],
            "completion": completion[:600],
            "stderr":     test_result["stderr"],
        })

    total     = len(results)
    pass_at_1 = passed_count / total if total else 0
    summary = {
        "label": label, "model": model_path,
        "total": total, "passed": passed_count, "pass_at_1": pass_at_1,
        "failed_ids": [r["task_id"] for r in results if not r["passed"]],
        "per_problem": {r["task_id"]: {"passed": r["passed"], "completion": r["completion"], "stderr": r["stderr"]} for r in results},
    }
    out = Path(RESULTS_DIR) / f"humaneval_{label}.json"
    out.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\n{'='*50}")
    print(f"  {label}: {passed_count}/{total} = {pass_at_1:.1%}")
    print(f"  Saved: {out}")
    print(f"{'='*50}\n")
    return summary

print("Eval functions ready.")

In [ ]:
# ── 先用 base 模型测 1 道题，确认 completion 和测试逻辑正常 ──────────
model, tokenizer = load_model(BASE_MODEL)

from datasets import load_dataset
prob0 = list(load_dataset("openai/openai_humaneval", split="test"))[0]

comp = generate_completion(model, tokenizer, prob0["prompt"])
res  = run_tests(prob0["prompt"], comp, prob0["test"], prob0["entry_point"])

print("=== Prompt ===")
print(prob0["prompt"])
print("\n=== Completion ===")
print(comp)
print("\n=== Test result ===")
print(res)

In [ ]:
# ── 正式评估三个模型（debug cell 确认正常后再跑）────────────────────
MODELS = [
    (BASE_MODEL,                                                                  "base"),
    ("Michlitt/codetune-v2-sft-A",                                               "sft_A"),
    (f"{DRIVE_ROOT}/sft_checkpoints/sft_C_with_targeted/final",                 "sft_C"),
]

all_results = {}
for model_id, label in MODELS:
    print(f"\n{'='*60}\nEvaluating: {label}\n{'='*60}")
    all_results[label] = evaluate(model_id, label)

In [ ]:
# ── 汇总表 ───────────────────────────────────────────────────────────
import json
from pathlib import Path

rows = []
for f in sorted(Path(RESULTS_DIR).glob("humaneval_*.json")):
    d = json.loads(f.read_text())
    rows.append(d)

print(f"{'Label':<12} {'pass@1':>8} {'passed':>10}")
print("-" * 34)
for r in rows:
    print(f"{r['label']:<12} {r['pass_at_1']:>8.1%} {r['passed']:>5}/{r['total']}")

# ── 8 个 targeted 问题对比 ────────────────────────────────────────────
TARGETED_IDS = [
    "HumanEval/54", "HumanEval/55", "HumanEval/107", "HumanEval/108",
    "HumanEval/110", "HumanEval/128", "HumanEval/141", "HumanEval/147"
]
labels = [r["label"] for r in rows]
data   = {r["label"]: r["per_problem"] for r in rows}

print(f"\n{'Task':<20}", end="")
for lb in labels:
    print(f"{lb:>10}", end="")
print()
print("-" * (20 + 10 * len(labels)))
for tid in TARGETED_IDS:
    print(f"{tid:<20}", end="")
    for lb in labels:
        passed = data.get(lb, {}).get(tid, {}).get("passed", None)
        print(f"{'✓' if passed else ('✗' if passed is False else '-'):>10}", end="")
    print()

In [ ]:
from google.colab import runtime
runtime.unassign()